[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_1/L2.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 1 | BÀI 2


---

# **MÔ HÌNH CÓ GIÁM SÁT: HỒI QUY VÀ SIÊU THAM SỐ**

|  |  |
|:---|:---|
|**Thời gian đọc** |  60 phút |
|**Kiến thức nền** | Hồi quy tuyến tính  
|**Từ khóa** | Hồi quy có phạt, mô hình tuyến tính, siêu tham số  |

---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
%pip install -q yfinance
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_1/"

*Trong bài này, ta giới thiệu các kỹ thuật chính để điều chuẩn hóa tham số, kiểm định chéo và điều chỉnh siêu tham số trong khuôn khổ hồi quy tuyến tính.*

## **1. Mô hình tuyến tính và quá khớp**

Trong bối cảnh hồi quy tuyến tính, một cách tốt để giảm quá khớp là điều chuẩn hóa (ràng buộc) các tham số, đôi khi còn gọi là "trọng số". Điều chuẩn hóa làm giảm bậc tự do của mô hình khi khớp dữ liệu huấn luyện.

Ở dạng tổng quát, gọi $J(\theta)$ là hàm chi phí trong quá trình huấn luyện của một mô hình có vector tham số $\theta$. Khi đó, điều chuẩn hóa được thực hiện bằng cách cộng thêm một số hạng phạt vào MSE của mô hình, nên hàm chi phí của hồi quy tuyến tính trở thành:
$$
\begin{align*}
J(\theta) = MSE(\theta) + Penalty(\theta)
\end{align*} 
$$

Tiếp theo ta mô tả ba phương pháp, gồm Ridge, Lasso và Elastic Net, mỗi phương pháp quy định số hạng phạt (điều chuẩn hóa) riêng để ràng buộc các trọng số. Một điểm quan trọng là ở phương pháp nào cũng vậy, số hạng phạt chỉ được cộng vào hàm chi phí trong lúc huấn luyện. Trên mẫu kiểm định hoặc mẫu kiểm tra, ta chỉ quan tâm đến hiệu suất của mô hình khi không điều chuẩn hóa.

### **1.1 Ridge**

Phương pháp Ridge là kỹ thuật điều chuẩn hóa dạng *co* (shrinkage), nhằm giữ các tham số của mô hình ở giá trị nhỏ nhất có thể. Để làm được vậy, số hạng điều chuẩn hóa có dạng
$$
\begin{align*}
\alpha\sum_{i=1}^n \theta_i^2
\end{align*}
$$
Siêu tham số $\alpha$ quyết định mức độ ta muốn điều chuẩn hóa mô hình. Nếu $\alpha=0$, hiển nhiên là ta không điều chuẩn hóa mô hình; còn nếu $\alpha$ rất lớn thì mọi tham số sẽ gần bằng không và đường khớp dữ liệu sẽ là một đường thẳng phẳng đi qua trung bình của các nhãn $y$ trong mẫu huấn luyện (lưu ý số hạng điều chuẩn hóa ở trên không chứa số hạng chệch $\theta_0$).

Hàm chi phí của hồi quy tuyến tính có phạt Ridge (hồi quy Ridge) khi đó là
$$
\begin{align*}
J(\theta) = MSE(\theta) + \frac{1}{2}\alpha\sum_{i=1}^n \theta_i^2
\end{align*}
$$

Do đó, nếu dùng gradient descent, bước cập nhật tham số sẽ là:
$$
\begin{align*}
\theta^{[next\ step]}=\theta^{[previous\ step]} - \eta \nabla_{\theta} MSE\left(\theta^{[previous\ step]}\right) - a \odot \theta^{[previous\ step]}
\end{align*}
$$

trong đó $a$ là vector cột gồm $n$ phần tử đều bằng $\alpha$, trừ phần tử đầu bằng 0 vì số hạng $\theta_0$ không được điều chuẩn hóa, và $\odot$ là phép nhân từng phần tử. Cuối cùng, lưu ý rằng hệ số $1/2$ nhân với $\alpha$ và các tham số trong hàm phạt chỉ nhằm cho đạo hàm gọn hơn khi tính gradient.

### **1.2 Lasso**

Phạt Lasso (Least Absolute Shrinkage and Selection Operator) là một phương pháp điều chuẩn hóa khác, có xu hướng loại bỏ trọng số của các đặc trưng ít quan trọng nhất bằng cách đưa chúng về không. Số hạng điều chuẩn hóa có dạng:
$$
\begin{align*}
\alpha\sum_{i=1}^n |\theta_i|
\end{align*} 
$$

Lưu ý số hạng này là chuẩn $\ell_1$, trong khi Ridge dùng chuẩn $\ell_2$ của vector tham số. Siêu tham số $\alpha$ một lần nữa quyết định tầm quan trọng chung của việc điều chuẩn hóa tham số. Hàm chi phí của hồi quy Lasso là:
$$
\begin{align*}
J(\theta) = MSE(\theta) + \alpha\sum_{i=1}^n |\theta_i|
\end{align*}
$$

Hồi quy Lasso thực hiện chọn đặc trưng và cho ra một mô hình thưa, chỉ có ít tham số khác không. Vì số hạng Lasso không khả vi tại $\theta_i=0$, thuật toán gradient descent cần được điều chỉnh phù hợp bằng phương pháp *dưới gradient* (subgradient):
$$
\begin{align*}
\theta^{[next\ step]}=\theta^{[previous\ step]} - \eta \nabla_{\theta} MSE\left(\theta^{[previous\ step]}\right) - a \odot \pmatrix{0 \\ sign(\theta_1)\\ \vdots \\ sign(\theta_n)}
\end{align*} 
$$

trong đó
$$
\begin{align*}
sign(\theta_i) = \begin{cases} -1\ \text{if}\ \theta_i < 0 \\
                                0\ \ \ \ \text{if}\ \theta_i = 0 \\
                                +1\ \text{if}\ \theta_i > 0 \end{cases}
\end{align*}
$$

### **1.3 Elastic Net**

Elastic Net kết hợp điều chuẩn hóa Ridge và Lasso trong một bài toán tối ưu duy nhất, trong đó một siêu tham số quyết định tỷ lệ kết hợp giữa co tham số và chọn đặc trưng khi huấn luyện mô hình. Hàm chi phí Elastic Net là:
$$
\begin{align*}
J(\theta) = MSE(\theta) + r\alpha\sum_{i=1}^n |\theta_i| + \frac{1-r}{2}\alpha\sum_{i=1}^n \theta_i^2
\end{align*}
$$

Siêu tham số $\alpha$ quyết định tầm quan trọng chung của việc điều chuẩn hóa tham số, còn siêu tham số $r$ cân bằng tỷ lệ pha trộn giữa chọn đặc trưng và co tham số.

Nên chọn phương pháp nào trong ba phương pháp? Ridge là lựa chọn mặc định tốt. Nhưng nếu ta ngờ rằng chỉ có vài đặc trưng thực sự hữu ích, nên chọn Lasso hoặc Elastic Net vì chúng loại bỏ những đặc trưng vô dụng. Nói chung, Lasso thuần có thể hoạt động thất thường khi số đặc trưng lớn hơn số quan sát huấn luyện, hoặc khi một số đặc trưng tương quan mạnh với nhau. Vì vậy Elastic Net là một lựa chọn tốt.

Lưu ý rằng trong hồi quy Elastic Net tối ưu bằng Gradient Descent, ta có ít nhất ba siêu tham số: tốc độ học $\eta$, mức điều chuẩn hóa chung của tham số $\alpha$, và tỷ trọng tương đối giữa điều chuẩn hóa Lasso và Ridge $r$. Tiếp theo, ta tìm cách xác định các giá trị "tốt" cho những siêu tham số này.

### **1.4 Hồi quy Elastic Net và xung lượng chuỗi thời gian**

Để minh họa cách hồi quy Elastic Net hoạt động, ta quay lại bài toán dự báo lợi suất cổ phiếu dựa trên thông tin quá khứ.

Trước hết ta nạp các thư viện cần thiết và định nghĩa ma trận đặc trưng đầu vào $X$ và nhãn $y$ như ở bài trước. Ta cũng chia mẫu thành tập huấn luyện và tập kiểm tra (tập kiểm định sẽ giới thiệu ở phần sau).

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

# Getting historical market data from SPY (ETF) (SPY)
df = yf.download("SPY", start="2000-01-01", end="2022-01-01")

In [ ]:
df["Ret"] = df["Adj Close"].pct_change()

name = "Ret"
df["Ret10_i"] = (
    df[name].rolling(10).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 10) - 1))
)
df["Ret25_i"] = (
    df[name].rolling(25).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 25) - 1))
)
df["Ret60_i"] = (
    df[name].rolling(60).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 60) - 1))
)
df["Ret120_i"] = (
    df[name].rolling(120).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 120) - 1))
)
df["Ret240_i"] = (
    df[name].rolling(240).apply(lambda x: 100 * ((np.prod(1 + x)) ** (1 / 240) - 1))
)

del df["Open"]
del df["Close"]
del df["High"]
del df["Low"]
del df["Volume"]
del df["Adj Close"]

df = df.dropna()
df.tail(10)

In [ ]:
df["Ret25"] = df["Ret25_i"].shift(-25)
df = df.dropna()
df.tail(10)

In [ ]:
X, y = df.iloc[:, 0:-1], df.iloc[:, -1]
print(X.shape, y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=int(len(y) * 0.5), shuffle=False
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Trước hết ta xét riêng hiệu suất của hồi quy Ridge và hồi quy Lasso. Hãy nhớ rằng hồi quy Ridge buộc thuật toán học không chỉ khớp dữ liệu mà còn giữ các tham số mô hình $\theta$ nhỏ nhất có thể.

Trong đoạn mã dưới đây, ta giữ cố định tham số điều chuẩn hóa $\alpha$ bằng 0,7.

In [ ]:
import matplotlib.pyplot as plt

# import ridge regression from sklearn library
from sklearn.linear_model import Ridge

In [ ]:
ridgeR = Ridge(alpha=0.7)
ridgeR.fit(X_train, y_train)
y_pred_ridge = ridgeR.predict(X_train)
y_pred_test_ridge = ridgeR.predict(X_test)

# calculate mean squared error
mean_squared_error = np.mean((y_pred_test_ridge - y_test) ** 2)
print("Mean squared error on test set", mean_squared_error)

In [ ]:
# get ridge coefficient and print them
ridge_coefficient = pd.DataFrame()
ridge_coefficient["Columns"] = X_train.columns
ridge_coefficient["Coefficient Estimate"] = pd.Series(ridgeR.coef_)
print(ridge_coefficient)

In [ ]:
# plotting the coefficient score
fig, ax = plt.subplots(figsize=(15, 7))

color = [
    "tab:gray",
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
    "tab:purple",
    "tab:brown",
    "tab:pink",
    "tab:gray",
    "tab:olive",
    "tab:cyan",
    "tab:orange",
    "tab:green",
    "tab:blue",
    "tab:olive",
]

ax.bar(
    ridge_coefficient["Columns"], ridge_coefficient["Coefficient Estimate"], color=color
)

plt.style.use("ggplot")
plt.show()

Hãy kiểm tra các tham số mô hình đã thay đổi thế nào khi huấn luyện với điều chuẩn hóa Ridge so với mô hình hồi quy tuyến tính đơn giản. Tham số có giá trị tuyệt đối lớn nhất trong hồi quy tuyến tính đã huấn luyện, tức tham số gắn với lợi suất ngày gần nhất, bị điều chỉnh giảm (xét theo giá trị tuyệt đối). Ngược lại, các tham số ở các độ trễ xa nhất tăng tầm quan trọng tương đối. Tuy nhiên, điều chuẩn hóa Ridge dường như bổ sung rất ít khả năng dự báo so với hồi quy tuyến tính (so với kết quả ở Bài 1).

Ta hãy xem cách áp dụng điều chuẩn hóa Lasso trong ứng dụng này.

In [ ]:
# import Lasso regression from sklearn library
from sklearn.linear_model import Lasso

# Train the model
lasso = Lasso(alpha=0.0001)
lasso.fit(X_train, y_train)
y_pred_Lasso = lasso.predict(X_train)
y_pred_test_lasso = lasso.predict(X_test)

# Calculate Mean Squared Error
mean_squared_error = np.mean((y_pred_test_lasso - y_test) ** 2)
print("Mean squared error on test set", mean_squared_error)

In [ ]:
lasso_coeff = pd.DataFrame()
lasso_coeff["Columns"] = X_train.columns
lasso_coeff["Coefficient Estimate"] = pd.Series(lasso.coef_)

print(lasso_coeff)

Lưu ý rằng điều chuẩn hóa Lasso co độ lớn của các tham số mạnh hơn so với hồi quy Ridge, nhưng không tham số nào về đúng bằng không. Do đó, thuật toán không coi đặc trưng đầu vào nào của mô hình là không liên quan. Tuy vậy, mô hình đã huấn luyện không tốt hơn phiên bản Ridge.

Bây giờ ta thử với mô hình hồi quy Elastic Net kết hợp cả hai kiểu điều chuẩn hóa. Lưu ý tham số "l1_ratio" trong đoạn mã dưới đây quyết định tầm quan trọng tương đối của Lasso so với Ridge trong mô hình.

Từ MSE ngoài mẫu, ta thấy việc kết hợp cả hai phương pháp điều chuẩn hóa đóng góp rất ít cho bài toán dự báo, đúng như ta kỳ vọng khi quan sát hiệu suất riêng của từng phương pháp.

In [ ]:
# import model
from sklearn.linear_model import ElasticNet

# Train the model
e_net = ElasticNet(alpha=0.0001, l1_ratio=0.1)
e_net.fit(X_train, y_train)

# calculate the prediction and mean square error
y_pred_elastic = e_net.predict(X_test)
mean_squared_error = np.mean((y_pred_elastic - y_test) ** 2)
print("Mean Squared Error on test set", mean_squared_error)

In [ ]:
e_net_coeff = pd.DataFrame()
e_net_coeff["Columns"] = X_train.columns
e_net_coeff["Coefficient Estimate"] = pd.Series(e_net.coef_)
e_net_coeff

## **2. Điều chỉnh siêu tham số**

Giả sử bây giờ bạn đã có một danh sách rút gọn các mô hình triển vọng. Việc cần làm tiếp theo là tinh chỉnh chúng. Ta hãy xem một vài cách làm.

### **2.1 Kiểm định chéo: GridSearchCV**

Một cách để biết giá trị tốt nhất của các siêu tham số là kiểm tra thủ công hiệu suất của mô hình với các siêu tham số khác nhau, nhưng cách này khá kém hiệu quả vì ta có thể không đủ thời gian để thử nhiều tổ hợp.

Một kỹ thuật hiệu quả hơn dựa trên kiểm định chéo: tập huấn luyện được chia thành các tập con bù nhau hay "fold". Với mỗi fold, mô hình được huấn luyện trên các fold còn lại với các siêu tham số khác nhau, còn fold đó được dùng làm tập kiểm định. Đây gọi là *kiểm định chéo $k$ phần* ($k$-fold cross-validation). Sau khi đã chọn được loại mô hình và siêu tham số, mô hình cuối cùng được huấn luyện với các siêu tham số này trên toàn bộ tập huấn luyện, và sai số khái quát hóa được đo trên tập kiểm tra.

GridSearchCV của Scikit-Learn sẽ thực hiện cả việc tinh chỉnh lẫn kiểm định chéo. Bạn chỉ cần cho nó biết muốn thử những siêu tham số nào và với những giá trị nào, nó sẽ đánh giá mọi tổ hợp giá trị siêu tham số có thể bằng kiểm định chéo.

### **2.2 Các cách điều chỉnh khác**

Phương pháp tìm kiếm lưới phù hợp khi ta khám phá tương đối ít tổ hợp, còn khi không gian tìm kiếm siêu tham số lớn, thường nên dùng *Tìm kiếm ngẫu nhiên* (Randomized Search) qua RandomizedSearchCV. Lớp này dùng gần giống GridSearchCV, nhưng thay vì thử mọi tổ hợp, nó đánh giá một số lượng tổ hợp siêu tham số ngẫu nhiên cho trước bằng cách chọn một giá trị ngẫu nhiên cho từng siêu tham số ở mỗi vòng lặp. Ưu điểm so với tìm kiếm lưới là mỗi lần chạy nó khám phá các giá trị siêu tham số khác nhau, thay vì các tổ hợp giá trị đặt sẵn từ trước.

Một cách tinh chỉnh hệ thống khác là thử kết hợp các mô hình tốt nhất. Nếu tổng hợp dự báo của một nhóm bộ dự báo (như các mô hình hồi quy khác nhau), ta thường được dự báo tốt hơn bộ dự báo đơn lẻ tốt nhất. Các *Phương pháp tổ hợp* (Ensemble Methods) cho phép kết hợp nhiều bộ dự báo từ các mô hình đã huấn luyện khác nhau để có một bộ dự báo tốt hơn nữa.

## **3. Hiệu suất khi điều chỉnh siêu tham số**

Đoạn mã dưới đây chạy bài tập kiểm định chéo kết hợp tìm kiếm lưới cho hồi quy Elastic Net để dự báo lợi suất cổ phiếu. Thuật toán $k$-fold chia ngẫu nhiên tập huấn luyện thành 10 tập con hay "fold" khác nhau, tương ứng tham số "n_splits" bên dưới. Sau đó, mỗi fold được dùng làm tập kiểm định trong khi mô hình được huấn luyện trên các fold còn lại. Như vậy ta có $k$ mô hình đã huấn luyện và có thể tính hiệu suất ước lượng của từng fold.

Vì $k$ fold được chia ngẫu nhiên, giá trị $k$ nhỏ có thể cho kết quả nhiễu. Nghĩa là mỗi lần chạy quy trình, tập dữ liệu có thể được chia thành $k$ fold theo cách khác, nên phân phối điểm hiệu suất có thể khác, dẫn đến ước lượng trung bình về hiệu suất của mô hình cũng khác. Để giải quyết, ta có thể lặp lại quy trình kiểm định chéo $k$ phần nhiều lần (tham số "n_repeat" bên dưới) và báo cáo hiệu suất trung bình trên tất cả các fold và các lần lặp. Cách tiếp cận này thường gọi là kiểm định chéo $k$ phần lặp lại.

*Lưu ý: tham số "random_state" đảm bảo thuật toán cho kết quả giống nhau, có thể tái lập, qua nhiều lần gọi hàm.*

Sau khi thiết lập kỹ thuật kiểm định chéo, ta đánh giá mô hình trên một lưới siêu tham số định trước của hồi quy Elastic Net, gồm mức điều chuẩn hóa chung $\alpha$ và tầm quan trọng tương đối của Lasso so với Ridge $r$. Ta xếp hạng các mô hình qua mỗi lần kiểm định chéo $k$ phần lặp lại theo MSE của chúng.

In [ ]:
from sklearn.model_selection import GridSearchCV, RepeatedKFold

model = ElasticNet()
# define model evaluation method
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
# define grid
grid = dict()
grid["alpha"] = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 0.0, 1.0, 10.0, 100.0]
grid["l1_ratio"] = [0, 0.01, 0.1, 0.2, 0.5, 0.7, 1]
search = GridSearchCV(model, grid, scoring="neg_mean_squared_error", cv=cv, n_jobs=-1)
# perform the search
results = search.fit(X_train, y_train)
# summarize
print("MSE: %.3f" % results.best_score_)
print("Config: %s" % results.best_params_)

Kết quả cho thấy mô hình tốt nhất từ kiểm định chéo rất gần với hồi quy tuyến tính thuần! Mô hình có một mức điều chuẩn hóa Ridge nhỏ và không có điều chuẩn hóa Lasso.

In [ ]:
# Train the model
e_net = ElasticNet(alpha=0.001, l1_ratio=0)  # Using the above hyperparameters
e_net.fit(X_train, y_train)

# calculate the prediction and mean square error
y_pred_elastic = e_net.predict(X_test)
mean_squared_error = np.mean((y_pred_elastic - y_test) ** 2)
print("Mean Squared Error on test set", mean_squared_error)

## **4. Kết luận**

Trong bài này, ta đã mô tả các phương pháp điều chuẩn hóa chính và các kỹ thuật tinh chỉnh để tìm tổ hợp siêu tham số tốt nhất trong mô hình hồi quy tuyến tính.

Trong bài tiếp theo, ta sẽ phân tích các đặc điểm chính của bài toán phân loại trong học máy.